# Ejecutor PPO con arranque por imitación, cuaderno B (semilla 2000)

La red del Ejecutor (PPO, la misma arquitectura y recompensa del Título I) **parte copiando a TWAP con órdenes límite**, la mejor política medida en ABIDES, y después **se ajusta con PPO** con la recompensa contra el precio de llegada, la penalización de 30 bps y γ = 1. La red recién imitada se guarda aparte para medir cuánto aporta PPO.

**Cómo usarlo:** «Ejecutar todas». Dura ≈ **1 h 15 min**. Si se corta, vuelve a ejecutar todo: retoma donde quedó. Los cuadernos A y B se pueden correr al mismo tiempo.

Queda en Drive en **`rimit/d3_opcion0_s2000`** (entrenada) y **`rimit_bc/d3_opcion0_s2000`** (solo imitación); al final se arma `rimit_B.zip`.

## Preparación del entorno (celdas 1 a 9)

In [ ]:
# CELDA 1 (reinicia el entorno solo; el aviso "sesión fallada" es normal, no la repitas)
!pip install -q "condacolab<0.2"
import condacolab
condacolab.install()

In [ ]:
# CELDA 2
import os
if not os.path.exists('/usr/local/envs/abides'):
    !conda create -n abides python=3.9 -y -q
!conda run -n abides python --version

In [ ]:
# CELDA 3 (unos minutos; un aviso de Pillow es normal)
%env PIP_DISABLE_PIP_VERSION_CHECK=1
!conda run -n abides python -m pip install -q pip==23.3.2 setuptools==57.5.0 wheel==0.38.4
!conda run -n abides python -m pip install -q "cython<3" numpy==1.22.0
!conda run -n abides python -m pip install -q --no-build-isolation --no-deps gym==0.18.0
!conda run -n abides python -m pip install -q --no-build-isolation numpy==1.22.0 "cloudpickle>=1.2,<1.7" "pyglet>=1.4,<=1.5.0" "Pillow<10" pandas==1.2.4 scipy==1.10.0 coloredlogs==15.0.1 psutil==5.8.0 tqdm==4.61.1 pomegranate==0.14.5 "ray[tune]==1.7.0" "protobuf==3.20.3" pyarrow==11.0.0 matplotlib==3.5.3 pytest
!conda run -n abides python -m pip install -q "gymnasium==0.29.1" "importlib-metadata>=4.8" farama-notifications --no-deps

In [ ]:
# CELDA 4 (deben salir tres "OK")
import os
if not os.path.exists('/content/abides-jpmc-public'):
    !git clone -q https://github.com/jpmorganchase/abides-jpmc-public.git /content/abides-jpmc-public
for pkg in ['abides-core', 'abides-markets', 'abides-gym']:
    !cd /content/abides-jpmc-public/{pkg} && conda run -n abides python setup.py -q install > /content/install_{pkg}.log 2>&1 && echo "{pkg} OK" || (echo "{pkg} FALLO"; tail -20 /content/install_{pkg}.log)

In [ ]:
# CELDA 5 (debe imprimir 0.18.0 1.22.0 1.2.4 0.29.1 1.7.0)
!conda run -n abides python -c "import gym, numpy, pandas, gymnasium, ray, abides_core, abides_markets, abides_gym; print(gym.__version__, numpy.__version__, pandas.__version__, gymnasium.__version__, ray.__version__)"

In [ ]:
# CELDA 6 (torch; debe imprimir torch 1.13.1+cpu | numpy 1.22.0)
!conda run -n abides python -m pip install -q torch==1.13.1+cpu --extra-index-url https://download.pytorch.org/whl/cpu
!conda run -n abides python -c "import torch, numpy; print('torch', torch.__version__, '| numpy', numpy.__version__)"

In [ ]:
# CELDA 7 (código del proyecto: rama feature/ps-cierre-sprint6; debe mostrar un commit y un número mayor que 0)
import os
if not os.path.exists('/content/slippage'):
    !git clone -q https://github.com/Vatelfil/slippage.git /content/slippage
!cd /content/slippage && git fetch -q origin && git checkout -q -f origin/feature/ps-cierre-sprint6
!cd /content/slippage && git log --oneline -1 && grep -c "imitar_heuristica" scripts/colab/entrenamiento_inicial_ps.py


In [ ]:
# CELDA 8 (conecta Drive; acepta el permiso)
from google.colab import drive
drive.mount('/content/drive')
RESULTS_DIR = '/content/drive/MyDrive/slippage_colab/resultados'
import os; os.makedirs(RESULTS_DIR, exist_ok=True)
print(RESULTS_DIR)

In [ ]:
# CELDA 9 (seaborn, que necesita el módulo de benchmarks)
!conda run -n abides python -m pip install -q seaborn==0.12.2 --no-deps

## Imitación y ajuste con PPO

In [ ]:
# Hiperparámetros: los de siempre. Si el resultado del barrido 3.1.1/3.1.2 ya está en la rama, se usa su ganador.
# La tasa de aprendizaje y la entropía se dejan bajas para no deshacer la imitación.
import json, os
ruta = '/content/slippage/data/barrido_hp/3_1_2/analisis.json'
c = {'minibatch_size': 64, 'epochs': 4, 'clip_ratio': 0.2, 'gae_lambda': 0.95}
if os.path.exists(ruta):
    c = json.load(open(ruta, encoding='utf-8'))['decision']['config']
HP = f"--minibatch {c['minibatch_size']} --epochs {c['epochs']} --clip {c['clip_ratio']} --gae-lambda {c['gae_lambda']} --lr 0.0001 --entropy-coef 0.001"
print(HP)


In [ ]:
# Imitación (100 episodios) + 1.000 episodios de PPO (≈ 1 h)
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/entrenamiento_inicial_ps.py --imitar twap_limit_medio --imitar-episodios 100 --guardar-imitacion {RESULTS_DIR}/rimit_bc/d3_opcion0_s2000 --recompensa is_llegada --penalizacion-bps 30 --gamma 1.0 {HP} --seed 2000 --episodes 1000 --out-dir {RESULTS_DIR}/rimit/d3_opcion0_s2000 --max-minutes 150 2>&1 | grep -v UserWarning


## Zip para descargar

In [ ]:
import os, zipfile
base = f'{RESULTS_DIR}/rimit/d3_opcion0_s2000'
archivos = [os.path.join(c, a) for c, _, fs in os.walk(base) if 'checkpoints' not in c for a in fs if a.endswith(('.json', '.csv'))]
zpath = f'{RESULTS_DIR}/rimit_B.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for a in archivos:
        z.write(a, os.path.relpath(a, RESULTS_DIR))
print(len(archivos), 'archivos -> zip listo:', zpath)
